# 02 · Computer-Vision Pipeline

Trains and runs the full CV stage end-to-end:

1. **Flower detector** (YOLO26, single class)
2. **Insect detector** (YOLO26, single class) + **pollinator/non-pollinator classifier** (iNaturalist-pretrained)
3. **Tracking + flower-visit counting** on the test videos

Heavy logic lives in `src/cv_engine/`; this notebook orchestrates it. It is
**idempotent** — any model whose weights already exist is loaded, not retrained,
so re-running is safe and fast. Requires the CV environment
(`bash scripts/setup_cv.sh`) and an NVIDIA GPU for training.

## Inlined source
Every `src` module below is embedded as an in-notebook module (no `from src` imports). Edit a module cell to change behaviour.

In [ ]:
import os, sys, json, glob
from pathlib import Path
import pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
try:
    import cv2, torch
except Exception:
    pass
_REPO = Path.cwd()
for _c in [_REPO, *_REPO.parents]:
    if (_c / "data").exists() and (_c / "src").exists(): _REPO = _c; break
os.chdir(_REPO)
# ---- src/config.py (inlined; edit here) ----
"""Central configuration for the Bee-A-Hero data pipeline.

All paths are anchored to the repository root (this file lives in ``src/``),
so the pipeline is portable across machines and does not depend on any
absolute/Windows path. Import this module rather than hard-coding paths.
"""

from pathlib import Path

# --- Repository layout -------------------------------------------------------
REPO_ROOT = _REPO

DATA_DIR: Path = REPO_ROOT / "data"
RAW_DIR: Path = DATA_DIR / "raw"
INTERIM_DIR: Path = DATA_DIR / "interim"
PROCESSED_DIR: Path = DATA_DIR / "processed"
BACKUP_DIR: Path = DATA_DIR / "_backup"

INAT_DIR: Path = RAW_DIR / "iNaturist"
FLOWER_DIR: Path = RAW_DIR / "Flower"
# Roboflow bee-detection COCO export (BEE.v8i). Place it here on any machine
# (git-ignored); overridable via the BEE_COCO_DIR env var for other layouts.
BEE_COCO_DIR: Path = Path(__import__("os").environ.get("BEE_COCO_DIR", RAW_DIR / "BEE_coco"))
# Videos to run the visit-counter on.
TEST_VIDEO_DIR: Path = RAW_DIR / "Test_Video"

# Published trained-weights repo on the Hugging Face Hub (for teammates to pull).
HF_WEIGHTS_REPO: str = "Manheim/bee-a-hero-cv"

# iNaturalist splits present on disk. ``public_test`` is unlabeled
# (annotations: 0) and is inference-only — it is NOT part of the labeled split.
INAT_LABELED_SPLITS: tuple[str, ...] = ("train_mini", "val")
INAT_UNLABELED_SPLIT: str = "public_test"

# Generated-artifact locations (git-ignored; see .gitignore data/interim/*).
MANIFEST_DIR: Path = INTERIM_DIR / "manifests"
EDA_DIR: Path = INTERIM_DIR / "eda"
REMOVED_DIR: Path = BACKUP_DIR / "removed"

# --- Reproducibility ---------------------------------------------------------
SEED: int = 42

# --- Split configuration -----------------------------------------------------
# Train/val/test proportions carved from the pooled labeled Insecta images.
SPLIT_RATIOS: dict[str, float] = {"train": 0.70, "val": 0.15, "test": 0.15}

# --- Taxonomy targets --------------------------------------------------------
# Keep only folders whose taxonomic Class == Insecta.
TARGET_CLASS: str = "Insecta"

# Bee families (Hymenoptera) tagged as a subset of interest for the project.
BEE_FAMILIES: frozenset[str] = frozenset({
    "Andrenidae", "Apidae", "Colletidae", "Halictidae",
    "Megachilidae", "Melittidae", "Stenotritidae",
})

# Valid image extensions.
IMAGE_EXTS: frozenset[str] = frozenset({".jpg", ".jpeg", ".png"})
from types import SimpleNamespace
C = SimpleNamespace(**{k: v for k, v in dict(globals()).items() if k.isupper()})
REPO = _REPO
print("repo:", REPO)

#### `src/data_pipeline/flower/make_detection_dataset.py` (inlined)

In [ ]:
# ===== src/data_pipeline/flower/make_detection_dataset.py  (inlined module — edit here) =====
import types as _t, sys as _s
mdd = _t.ModuleType("mdd")
mdd.C = C
_src = r'''
"""
make_detection_dataset.py
-------------------------
Turn the merged Flower Classification dataset (class subfolders) into an
OBJECT-DETECTION-ready dataset in YOLO format, AND emit a labels.csv that
maps every image to its class.

IMPORTANT / HONEST NOTE
-----------------------
The source is a *classification* dataset: one centered flower per image, with
NO ground-truth boxes. This script GENERATES boxes automatically with OpenCV
GrabCut foreground segmentation (a tight box around the detected flower region).
These boxes are approximate and machine-made, not human-verified. They give you
"clear boundaries" to train/prototype a detector, but you should spot-check and
hand-correct a sample before trusting them as ground truth.

INPUT  (created by merge_flowers.py):
    merged_dataset/
        Training Data/<Class>/*.jpeg
        Validation Data/<Class>/*.jpeg
        Testing Data/<Class>/*.jpeg

OUTPUT:
    yolo_dataset/
        images/{train,val,test}/<Class>_<orig>.jpeg
        labels/{train,val,test}/<Class>_<orig>.txt     # "<cls> cx cy w h" (normalized)
        data.yaml
        classes.txt
    labels.csv         # image -> class mapping (classification)
    annotations.csv    # image -> class + pixel bbox (detection, human-readable)

Usage:
    python make_detection_dataset.py
    python make_detection_dataset.py --src merged_dataset --out yolo_dataset --workers 16
    python make_detection_dataset.py --limit 20    # quick smoke test (20 imgs/class)
"""

import argparse
import csv
import os
import shutil
from concurrent.futures import ProcessPoolExecutor, as_completed
from pathlib import Path

import cv2
import numpy as np

IMAGE_EXTS = {".jpeg", ".jpg", ".png"}
SPLIT_MAP = {
    "Training Data": "train",
    "Validation Data": "val",
    "Testing Data": "test",
}
GC_SIZE = 160          # work resolution for GrabCut (speed)
GC_ITERS = 3           # GrabCut iterations
MIN_AREA_FRAC = 0.02   # if fg smaller than this -> fallback box
MAX_AREA_FRAC = 0.985  # if fg bigger than this  -> fallback box
FALLBACK_FRAC = 0.92   # centered fallback box size (fraction of image)


def auto_bbox(img):
    """Return (cx,cy,w,h) normalized [0,1] for the main flower region.
    Falls back to a centered box if segmentation is unreliable."""
    h, w = img.shape[:2]
    scale = GC_SIZE / max(h, w)
    sw, sh = max(1, int(w * scale)), max(1, int(h * scale))
    small = cv2.resize(img, (sw, sh))

    mask = np.zeros((sh, sw), np.uint8)
    bgd = np.zeros((1, 65), np.float64)
    fgd = np.zeros((1, 65), np.float64)
    m = 0.06  # border margin -> assumed background
    rect = (int(sw * m), int(sh * m), int(sw * (1 - 2 * m)), int(sh * (1 - 2 * m)))

    used_fallback = False
    try:
        cv2.grabCut(small, mask, rect, bgd, fgd, GC_ITERS, cv2.GC_INIT_WITH_RECT)
        fg = np.where((mask == cv2.GC_FGD) | (mask == cv2.GC_PR_FGD), 255, 0).astype("uint8")
        fg = cv2.morphologyEx(fg, cv2.MORPH_CLOSE, np.ones((5, 5), np.uint8))
        cnts, _ = cv2.findContours(fg, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if cnts:
            c = max(cnts, key=cv2.contourArea)
            x, y, bw, bh = cv2.boundingRect(c)
            area_frac = (bw * bh) / float(sw * sh)
            if not (MIN_AREA_FRAC <= area_frac <= MAX_AREA_FRAC):
                used_fallback = True
        else:
            used_fallback = True
    except Exception:
        used_fallback = True

    if used_fallback:
        f = FALLBACK_FRAC
        cx, cy, nw, nh = 0.5, 0.5, f, f
    else:
        cx = (x + bw / 2) / sw
        cy = (y + bh / 2) / sh
        nw = bw / sw
        nh = bh / sh

    # clamp
    cx, cy = min(max(cx, 0), 1), min(max(cy, 0), 1)
    nw, nh = min(nw, 1), min(nh, 1)
    return cx, cy, nw, nh, used_fallback


def process_one(task):
    """Worker: compute bbox for a single image. Returns dict or None on read error."""
    src, split, cls, cls_id, dst_name = task
    img = cv2.imread(src)
    if img is None:
        return None
    h, w = img.shape[:2]
    cx, cy, nw, nh, fb = auto_bbox(img)
    # pixel coords for the human-readable annotations.csv
    bw_px, bh_px = nw * w, nh * h
    xmin = int(round(cx * w - bw_px / 2))
    ymin = int(round(cy * h - bh_px / 2))
    xmax = int(round(cx * w + bw_px / 2))
    ymax = int(round(cy * h + bh_px / 2))
    xmin, ymin = max(0, xmin), max(0, ymin)
    xmax, ymax = min(w, xmax), min(h, ymax)
    return {
        "src": src, "split": split, "cls": cls, "cls_id": cls_id,
        "dst_name": dst_name, "w": w, "h": h,
        "cx": cx, "cy": cy, "nw": nw, "nh": nh,
        "xmin": xmin, "ymin": ymin, "xmax": xmax, "ymax": ymax,
        "fallback": fb,
    }


def build_tasks(src_root, limit):
    classes = set()
    tasks = []
    per_class_counter = {}
    for split_dir, split_key in SPLIT_MAP.items():
        sdir = src_root / split_dir
        if not sdir.is_dir():
            continue
        for cls_dir in sorted(p for p in sdir.iterdir() if p.is_dir()):
            cls = cls_dir.name
            classes.add(cls)
    class_list = sorted(classes)
    class_id = {c: i for i, c in enumerate(class_list)}

    seen_names = set()
    for split_dir, split_key in SPLIT_MAP.items():
        sdir = src_root / split_dir
        if not sdir.is_dir():
            continue
        for cls_dir in sorted(p for p in sdir.iterdir() if p.is_dir()):
            cls = cls_dir.name
            key = (split_key, cls)
            per_class_counter.setdefault(key, 0)
            for f in sorted(cls_dir.iterdir()):
                if f.suffix.lower() not in IMAGE_EXTS:
                    continue
                if limit and per_class_counter[key] >= limit:
                    break
                per_class_counter[key] += 1
                # unique, collision-proof destination name (flat per split).
                # Dedup on the STEM (not full name) so that e.g. "X.jpeg" and
                # "X.png" don't collide onto the same label .txt file.
                base = f"{cls}_{f.name}"
                stem = Path(base).stem
                dst = base
                i = 1
                while (split_key, Path(dst).stem) in seen_names:
                    dst = f"{stem}_{i}{f.suffix}"
                    i += 1
                seen_names.add((split_key, Path(dst).stem))
                tasks.append((str(f), split_key, cls, class_id[cls], dst))
    return tasks, class_list, class_id


# repo root = .../Bee-A-Hero  (this file is at src/data_pipeline/flower/make_detection_dataset.py)
REPO = Path.cwd()
DEFAULT_SRC = REPO / "data" / "processed" / "flower" / "classification"
DEFAULT_OUT = REPO / "data" / "processed" / "flower" / "yolo"


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--src", default=str(DEFAULT_SRC),
                    help="classification dataset (default: data/processed/flower/classification)")
    ap.add_argument("--out", default=str(DEFAULT_OUT),
                    help="YOLO output (default: data/processed/flower/yolo)")
    ap.add_argument("--workers", type=int, default=max(1, (os.cpu_count() or 4) - 2))
    ap.add_argument("--limit", type=int, default=0, help="max images per class+split (0 = all)")
    args = ap.parse_args()

    src_root = Path(args.src)
    out = Path(args.out)
    for split in ("train", "val", "test"):
        (out / "images" / split).mkdir(parents=True, exist_ok=True)
        (out / "labels" / split).mkdir(parents=True, exist_ok=True)

    tasks, class_list, class_id = build_tasks(src_root, args.limit)
    print(f"Images to process: {len(tasks)}  |  classes: {len(class_list)}  |  workers: {args.workers}")

    labels_rows = []       # image_path, split, class, class_id
    ann_rows = []          # image_path, width, height, class, class_id, xmin, ymin, xmax, ymax
    fallback_count = 0
    done = 0

    with ProcessPoolExecutor(max_workers=args.workers) as ex:
        futs = [ex.submit(process_one, t) for t in tasks]
        for fut in as_completed(futs):
            r = fut.result()
            done += 1
            if r is None:
                continue
            if r["fallback"]:
                fallback_count += 1

            split = r["split"]
            dst_img = out / "images" / split / r["dst_name"]
            dst_lbl = out / "labels" / split / (Path(r["dst_name"]).stem + ".txt")

            shutil.copy2(r["src"], dst_img)
            with open(dst_lbl, "w") as fh:
                fh.write(f"{r['cls_id']} {r['cx']:.6f} {r['cy']:.6f} {r['nw']:.6f} {r['nh']:.6f}\n")

            rel = f"images/{split}/{r['dst_name']}"
            labels_rows.append([rel, split, r["cls"], r["cls_id"]])
            ann_rows.append([rel, r["w"], r["h"], r["cls"], r["cls_id"],
                             r["xmin"], r["ymin"], r["xmax"], r["ymax"]])

            if done % 2000 == 0:
                print(f"  ...{done}/{len(tasks)}")

    # sort for stable output
    labels_rows.sort()
    ann_rows.sort()

    with open(out / "labels.csv", "w", newline="") as fh:
        w = csv.writer(fh)
        w.writerow(["image", "split", "class", "class_id"])
        w.writerows(labels_rows)

    with open(out / "annotations.csv", "w", newline="") as fh:
        w = csv.writer(fh)
        w.writerow(["image", "width", "height", "class", "class_id",
                    "xmin", "ymin", "xmax", "ymax"])
        w.writerows(ann_rows)

    with open(out / "classes.txt", "w") as fh:
        fh.write("\n".join(class_list) + "\n")

    with open(out / "data.yaml", "w") as fh:
        fh.write(f"path: {out.resolve()}\n")
        fh.write("train: images/train\n")
        fh.write("val: images/val\n")
        fh.write("test: images/test\n")
        fh.write(f"nc: {len(class_list)}\n")
        fh.write("names:\n")
        for c in class_list:
            fh.write(f"  - {c}\n")

    print("\n" + "=" * 60)
    print("DETECTION DATASET READY ->", out.resolve())
    print("=" * 60)
    print(f"images written : {len(labels_rows)}")
    print(f"fallback boxes : {fallback_count} "
          f"({100*fallback_count/max(1,len(labels_rows)):.1f}% used centered box)")
    print(f"classes        : {class_list}")
    print("files          : data.yaml, classes.txt, labels.csv, annotations.csv")


if __name__ == "__main__":
    main()
'''
exec(compile(_src, "src/data_pipeline/flower/make_detection_dataset.py", "exec"), mdd.__dict__)
_s.modules["mdd"] = mdd
globals()["mdd"] = mdd

#### `src/cv_engine/sam_box.py` (inlined)

In [ ]:
# ===== src/cv_engine/sam_box.py  (inlined module — edit here) =====
import types as _t, sys as _s
sam_box = _t.ModuleType("sam_box")
sam_box.C = C
_src = r'''
"""Tight bounding boxes for centered iNaturalist organisms via SAM.

iNaturalist crops show one organism, roughly centered. Prompting Segment
Anything (MobileSAM, bundled with Ultralytics — no extra dependency) with a
center point yields a tight mask of just the insect, whose bounding box hugs the
animal far better than GrabCut's near-full-frame boxes.

``sam_box(path)`` returns ``(cx, cy, w, h)`` normalized [0,1], or a centered
fallback if SAM finds nothing usable. The SAM model is loaded once (lazy).
"""

from functools import lru_cache

import numpy as np

_FALLBACK = (0.5, 0.5, 0.85, 0.85)


@lru_cache(maxsize=1)
def _model():
    from ultralytics import SAM
    return SAM("mobile_sam.pt")


def sam_box(path: str, min_frac: float = 0.01, max_frac: float = 0.999):
    """Tight (cx, cy, w, h) normalized box around the centered organism."""
    import cv2
    img = cv2.imread(str(path))
    if img is None:
        return _FALLBACK
    h, w = img.shape[:2]
    r = _model().predict(img, points=[[w / 2, h / 2]], labels=[1], verbose=False)[0]
    if r.masks is None or len(r.masks) == 0:
        return _FALLBACK
    mask = r.masks.data[0].cpu().numpy().astype("uint8")
    ys, xs = np.where(mask > 0)
    if xs.size == 0:
        return _FALLBACK
    x1, x2, y1, y2 = xs.min(), xs.max(), ys.min(), ys.max()
    bw, bh = (x2 - x1) / w, (y2 - y1) / h
    if not (min_frac <= bw * bh <= max_frac):
        return _FALLBACK
    cx = (x1 + x2) / 2 / w
    cy = (y1 + y2) / 2 / h
    return (float(cx), float(cy), float(bw), float(bh))
'''
exec(compile(_src, "src/cv_engine/sam_box.py", "exec"), sam_box.__dict__)
_s.modules["sam_box"] = sam_box
globals()["sam_box"] = sam_box

#### `src/cv_engine/prepare_flower.py` (inlined)

In [ ]:
# ===== src/cv_engine/prepare_flower.py  (inlined module — edit here) =====
import types as _t, sys as _s
prepare_flower = _t.ModuleType("prepare_flower")
prepare_flower.C = C
prepare_flower.auto_bbox = mdd.auto_bbox
_src = r'''
"""Build a single-class ``flower`` YOLO detection dataset from the Flower
*classification* folders (data/raw/Flower).

The source has no boxes (one centered flower per image), so boxes are generated
with OpenCV GrabCut foreground segmentation — reusing the proven ``auto_bbox``
from ``src/data_pipeline/flower/make_detection_dataset.py`` (no duplicated code).
All boxes are class 0 = ``flower`` (per the CV plan). Approximate/machine-made —
spot-check before trusting as ground truth.

Output (git-ignored, under data/interim/):
    flower_det/images/{train,val,test}/<Class>_<file>
    flower_det/labels/{train,val,test}/<Class>_<file>.txt   # "0 cx cy w h"
    flower_det/data.yaml                                     # Ultralytics config

CLI:  python -m src.cv_engine.prepare_flower
"""

import multiprocessing
import shutil
from concurrent.futures import ProcessPoolExecutor
from pathlib import Path

import cv2


_MP = multiprocessing.get_context("fork")
SPLIT_MAP = {"Training Data": "train", "Validation Data": "val", "Testing Data": "test"}
IMG_EXTS = {".jpg", ".jpeg", ".png"}
OUT = C.INTERIM_DIR / "flower_det"


def _build_tasks() -> list[tuple[str, str, str]]:
    tasks, seen = [], set()
    for split_dir, split in SPLIT_MAP.items():
        sdir = C.FLOWER_DIR / split_dir
        if not sdir.is_dir():
            continue
        for cls_dir in sorted(p for p in sdir.iterdir() if p.is_dir()):
            for f in sorted(cls_dir.iterdir()):
                if f.suffix.lower() not in IMG_EXTS:
                    continue
                stem, dst, i = f"{cls_dir.name}_{f.stem}", f"{cls_dir.name}_{f.name}", 1
                while (split, Path(dst).stem) in seen:
                    dst = f"{stem}_{i}{f.suffix}"; i += 1
                seen.add((split, Path(dst).stem))
                tasks.append((str(f), split, dst))
    return tasks


def _process(task):
    src, split, dst = task
    img = cv2.imread(src)
    if img is None:
        return (split, "read_error")
    cx, cy, nw, nh, fb = auto_bbox(img)
    shutil.copy2(src, OUT / "images" / split / dst)
    (OUT / "labels" / split / f"{Path(dst).stem}.txt").write_text(
        f"0 {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}\n")
    return (split, "fallback" if fb else "grabcut")


def _write_yaml() -> None:
    (OUT / "data.yaml").write_text(
        f"# single-class flower detector (GrabCut auto-boxes)\n"
        f"path: {OUT.resolve()}\n"
        f"train: images/train\nval: images/val\ntest: images/test\n"
        f"nc: 1\nnames: [flower]\n")


def run(workers: int | None = None) -> dict:
    for split in ("train", "val", "test"):
        (OUT / "images" / split).mkdir(parents=True, exist_ok=True)
        (OUT / "labels" / split).mkdir(parents=True, exist_ok=True)
    tasks = _build_tasks()
    from collections import Counter
    stats: Counter = Counter()
    with ProcessPoolExecutor(max_workers=workers, mp_context=_MP) as ex:
        for split, kind in ex.map(_process, tasks, chunksize=32):
            stats[f"{split}_{kind}"] += 1
            stats[split] += 1
    _write_yaml()
    summary = {"total": len(tasks), "counts": dict(stats), "data_yaml": str(OUT / "data.yaml")}
    return summary


if __name__ == "__main__":
    import json, os
    print(json.dumps(run(workers=os.cpu_count()), indent=2))
'''
exec(compile(_src, "src/cv_engine/prepare_flower.py", "exec"), prepare_flower.__dict__)
_s.modules["prepare_flower"] = prepare_flower
globals()["prepare_flower"] = prepare_flower

#### `src/cv_engine/prepare_insect.py` (inlined)

In [ ]:
# ===== src/cv_engine/prepare_insect.py  (inlined module — edit here) =====
import types as _t, sys as _s
prepare_insect = _t.ModuleType("prepare_insect")
prepare_insect.C = C
prepare_insect.auto_bbox = mdd.auto_bbox
_src = r'''
"""Build a 2-class insect YOLO detection dataset: pollinator vs non_pollinator.

iNaturalist is a *classification* set (no boxes), so boxes are generated with
GrabCut (reusing ``auto_bbox``) on the centered organism. Classes come from the
Phase-4 manifest ``is_bee`` flag (bee families = pollinator). Non-pollinator is
heavily over-represented (147k vs 3.7k), so it is sub-sampled per split (seeded)
to balance 1:1 with pollinator.

Real bee boxes from the Roboflow COCO sets on the Desktop (iNat-sourced +
video-frame bees) are converted to YOLO and added to the **pollinator** class to
improve mAP and robustness on real video.

Classes: 0 = pollinator, 1 = non_pollinator.

Output (git-ignored): data/interim/insect_det/{images,labels}/{train,val,test}
                      + data.yaml

CLI:  python -m src.cv_engine.prepare_insect
"""

import csv
import json
import multiprocessing
import random
import shutil
from collections import Counter, defaultdict
from concurrent.futures import ProcessPoolExecutor
from pathlib import Path

import cv2


_MP = multiprocessing.get_context("fork")
OUT = C.INTERIM_DIR / "insect_det"
NAMES = ["pollinator", "non_pollinator"]

# Real-box bee detection set (Roboflow COCO export).
# Only BEE.v8i (single iNat-sourced garden bees) — the Honey Bee hive export
# (~16 tiny dense bees/frame, hive-monitoring domain) tanks detection mAP and
# does not match the garden flower-visit videos, so it is excluded from detection.
# Portable location (git-ignored): data/raw/BEE_coco/ (see src/config.BEE_COCO_DIR).
# If absent, the pipeline still runs on iNaturalist alone (roboflow step skipped).
ROBOFLOW_SETS = [
    C.BEE_COCO_DIR,
]
_RF_SPLIT = {"train": "train", "valid": "val", "test": "test"}


# --------------------------------------------------------------------------- #
# iNaturalist -> GrabCut boxes, 2 classes, balanced
# --------------------------------------------------------------------------- #
def _select_inat_records() -> list[tuple[str, str, int, str]]:
    """Return (abs_path, split, class_id, dst_name); non_pollinator balanced 1:1."""
    rows = list(csv.DictReader(open(C.MANIFEST_DIR / "split_manifest.csv")))
    by_split_cls: dict[tuple[str, int], list[dict]] = defaultdict(list)
    for r in rows:
        cls = 0 if r["is_bee"] == "1" else 1
        by_split_cls[(r["split"], cls)].append(r)

    rng = random.Random(C.SEED)
    out = []
    for split in ("train", "val", "test"):
        poll = by_split_cls[(split, 0)]
        non = by_split_cls[(split, 1)]
        # balance: keep all pollinators, sample equal non-pollinators
        non = sorted(non, key=lambda r: r["path"])
        rng.shuffle(non)
        non = non[: len(poll)]
        for cls, recs in ((0, poll), (1, non)):
            for r in recs:
                p = C.REPO_ROOT / r["path"]
                dst = f"inat_{Path(r['path']).stem}.jpg"
                out.append((str(p), split, cls, dst))
    return out


def _process_inat(task):
    src, split, cls, dst = task
    img = cv2.imread(src)
    if img is None:
        return (split, "read_error")
    cx, cy, nw, nh, fb = auto_bbox(img)
    shutil.copy2(src, OUT / "images" / split / dst)
    (OUT / "labels" / split / f"{Path(dst).stem}.txt").write_text(
        f"{cls} {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}\n")
    return (split, f"cls{cls}")


# --------------------------------------------------------------------------- #
# Roboflow COCO -> YOLO (all boxes -> class 0 pollinator)
# --------------------------------------------------------------------------- #
def _convert_roboflow() -> Counter:
    stats: Counter = Counter()
    for ds in ROBOFLOW_SETS:
        if not ds.is_dir():
            continue
        tag = ds.name.split(".")[0].replace(" ", "_")[:20]
        for rf_split, our_split in _RF_SPLIT.items():
            ann = ds / rf_split / "_annotations.coco.json"
            if not ann.exists():
                continue
            coco = json.load(open(ann))
            img_by_id = {im["id"]: im for im in coco["images"]}
            boxes_by_img: dict[int, list] = defaultdict(list)
            for a in coco["annotations"]:
                boxes_by_img[a["image_id"]].append(a["bbox"])  # [x,y,w,h] pixels
            for iid, im in img_by_id.items():
                src = ds / rf_split / im["file_name"]
                if not src.exists():
                    continue
                W, H = im["width"], im["height"]
                lines = []
                for x, y, w, h in boxes_by_img.get(iid, []):
                    cx, cy = (x + w / 2) / W, (y + h / 2) / H
                    lines.append(f"0 {cx:.6f} {cy:.6f} {w / W:.6f} {h / H:.6f}")
                if not lines:                      # skip images with no bee box
                    continue
                dst = f"rf_{tag}_{our_split}_{iid}.jpg"
                shutil.copy2(src, OUT / "images" / our_split / dst)
                (OUT / "labels" / our_split / f"{Path(dst).stem}.txt").write_text(
                    "\n".join(lines) + "\n")
                stats[f"{our_split}_roboflow"] += 1
    return stats


def export_single_class(dst: Path | None = None) -> str:
    """Derive a single-class ``insect`` detection set from the 2-class boxes.

    Reuses the GrabCut/Roboflow boxes already in ``insect_det`` (no re-compute):
    images are symlinked, labels rewritten to class 0. Used to train the
    high-mAP detector; the 2-class set stays for building classifier crops.
    """
    dst = dst or (C.INTERIM_DIR / "insect_det1")
    for split in ("train", "val", "test"):
        (dst / "images" / split).mkdir(parents=True, exist_ok=True)
        (dst / "labels" / split).mkdir(parents=True, exist_ok=True)
        for img in (OUT / "images" / split).iterdir():
            link = dst / "images" / split / img.name
            if not link.exists():
                link.symlink_to(img.resolve())
        for lbl in (OUT / "labels" / split).glob("*.txt"):
            lines = ["0 " + " ".join(l.split()[1:])
                     for l in lbl.read_text().splitlines() if l.strip()]
            (dst / "labels" / split / lbl.name).write_text("\n".join(lines) + "\n")
    (dst / "data.yaml").write_text(
        f"# single-class insect detector (high-mAP stage 1)\n"
        f"path: {dst.resolve()}\ntrain: images/train\nval: images/val\ntest: images/test\n"
        f"nc: 1\nnames: [insect]\n")
    return str(dst / "data.yaml")


def export_classifier_crops(dst: Path | None = None, min_size: int = 20) -> dict:
    """Crop every labeled box from the 2-class set into an ImageFolder for the
    pollinator/non_pollinator classifier: ``insect_cls/<split>/<class>/<crop>.jpg``.
    """
    dst = dst or (C.INTERIM_DIR / "insect_cls")
    names = {0: "pollinator", 1: "non_pollinator"}
    counts: Counter = Counter()
    for split in ("train", "val", "test"):
        for cn in names.values():
            (dst / split / cn).mkdir(parents=True, exist_ok=True)
        for lbl in sorted((OUT / "labels" / split).glob("*.txt")):
            cand = list((OUT / "images" / split).glob(lbl.stem + ".*"))
            if not cand:
                continue
            img = cv2.imread(str(cand[0]))
            if img is None:
                continue
            H, W = img.shape[:2]
            for i, line in enumerate(lbl.read_text().splitlines()):
                p = line.split()
                if len(p) < 5:
                    continue
                c = int(p[0]); cx, cy, w, h = map(float, p[1:5])
                x1, y1 = max(0, int((cx - w / 2) * W)), max(0, int((cy - h / 2) * H))
                x2, y2 = min(W, int((cx + w / 2) * W)), min(H, int((cy + h / 2) * H))
                if x2 - x1 < min_size or y2 - y1 < min_size:
                    continue
                cv2.imwrite(str(dst / split / names[c] / f"{lbl.stem}_{i}.jpg"),
                            img[y1:y2, x1:x2])
                counts[f"{split}_{names[c]}"] += 1
    return dict(counts)


def _write_yaml() -> None:
    (OUT / "data.yaml").write_text(
        f"# 2-class insect detector: pollinator vs non_pollinator\n"
        f"path: {OUT.resolve()}\n"
        f"train: images/train\nval: images/val\ntest: images/test\n"
        f"nc: 2\nnames: [pollinator, non_pollinator]\n")


def run(workers: int | None = None) -> dict:
    for split in ("train", "val", "test"):
        (OUT / "images" / split).mkdir(parents=True, exist_ok=True)
        (OUT / "labels" / split).mkdir(parents=True, exist_ok=True)

    tasks = _select_inat_records()
    stats: Counter = Counter()
    with ProcessPoolExecutor(max_workers=workers, mp_context=_MP) as ex:
        for split, kind in ex.map(_process_inat, tasks, chunksize=32):
            stats[f"inat_{split}_{kind}"] += 1
    stats.update(_convert_roboflow())
    _write_yaml()
    return {"inat_images": len(tasks), "counts": dict(stats),
            "data_yaml": str(OUT / "data.yaml")}


if __name__ == "__main__":
    import os
    print(json.dumps(run(workers=os.cpu_count()), indent=2))
'''
exec(compile(_src, "src/cv_engine/prepare_insect.py", "exec"), prepare_insect.__dict__)
_s.modules["prepare_insect"] = prepare_insect
globals()["prepare_insect"] = prepare_insect

#### `src/cv_engine/train.py` (inlined)

In [ ]:
# ===== src/cv_engine/train.py  (inlined module — edit here) =====
import types as _t, sys as _s
cvtrain = _t.ModuleType("cvtrain")
cvtrain.C = C
_src = r'''
"""Fine-tune a YOLO26 detector (flowers or insects) on the RTX 3050 6GB.

Thin, reusable wrapper over Ultralytics. Runs are written under
``data/interim/cv_runs/<name>/`` (git-ignored); best weights at
``.../weights/best.pt``.

CLI:
    python -m src.cv_engine.train --data data/interim/flower_det/data.yaml \
        --name flower_yolo26n --epochs 60 --batch 16
"""

import argparse
import multiprocessing
from pathlib import Path

# Python 3.14 defaults to the "forkserver" start method, which makes Ultralytics'
# DataLoader workers re-import this module and spawn a runaway process swarm with
# no real training. Force "fork" so workers inherit state cleanly.
try:
    multiprocessing.set_start_method("fork", force=True)
except (RuntimeError, ValueError):
    pass  # fork unavailable (Windows) -> use the default start method

from ultralytics import YOLO


RUNS_DIR = C.INTERIM_DIR / "cv_runs"
WEIGHTS_DIR = C.INTERIM_DIR / "weights"


def train(data: str, name: str, model: str = "yolo26n.pt", epochs: int = 60,
          imgsz: int = 640, batch: int = 16, device: int | str = 0,
          patience: int = 15, resume: bool = False, scale: float = 0.5) -> dict:
    RUNS_DIR.mkdir(parents=True, exist_ok=True)
    # prefer a local pretrained copy if present (avoids re-download to cwd)
    local = WEIGHTS_DIR / model
    yolo = YOLO(str(local) if local.exists() else model)
    results = yolo.train(
        data=data, epochs=epochs, imgsz=imgsz, batch=batch, device=device,
        project=str(RUNS_DIR), name=name, seed=C.SEED, deterministic=True,
        amp=True, patience=patience, exist_ok=True, resume=resume, verbose=True,
        # scale jitter: larger range makes the detector see objects at more
        # scales (incl. small) -> tighter boxes on small/video insects.
        scale=scale,
    )
    best = RUNS_DIR / name / "weights" / "best.pt"
    # report validation mAP
    metrics = yolo.val(data=data, imgsz=imgsz, device=device, verbose=False)
    out = {
        "name": name, "best_weights": str(best),
        "map50": round(float(metrics.box.map50), 4),
        "map50_95": round(float(metrics.box.map), 4),
    }
    return out


def main() -> None:
    ap = argparse.ArgumentParser(description=__doc__)
    ap.add_argument("--data", required=True, help="path to data.yaml")
    ap.add_argument("--name", required=True, help="run name")
    ap.add_argument("--model", default="yolo26n.pt")
    ap.add_argument("--epochs", type=int, default=60)
    ap.add_argument("--imgsz", type=int, default=640)
    ap.add_argument("--batch", type=int, default=16)
    ap.add_argument("--patience", type=int, default=15)
    ap.add_argument("--resume", action="store_true")
    ap.add_argument("--scale", type=float, default=0.5, help="scale-jitter gain (aug)")
    args = ap.parse_args()
    import json
    print(json.dumps(train(args.data, args.name, args.model, args.epochs,
                           args.imgsz, args.batch, patience=args.patience,
                           resume=args.resume, scale=args.scale), indent=2))


if __name__ == "__main__":
    main()
'''
exec(compile(_src, "src/cv_engine/train.py", "exec"), cvtrain.__dict__)
_s.modules["cvtrain"] = cvtrain
globals()["cvtrain"] = cvtrain

#### `src/cv_engine/train_classifier.py` (inlined)

In [ ]:
# ===== src/cv_engine/train_classifier.py  (inlined module — edit here) =====
import types as _t, sys as _s
train_classifier = _t.ModuleType("train_classifier")
train_classifier.C = C
_src = r'''
"""Fine-tune an iNaturalist-pretrained classifier for pollinator vs non_pollinator.

Stage 2 of the two-stage insect pipeline. Because the crops are iNaturalist-style
organisms, an iNat21-pretrained backbone (timm) already separates bees from other
insects extremely well — we replace its 10k-class head with a 2-class head and
fine-tune (optionally head-only / linear-probe for big models on a 6GB card).

Reads the ImageFolder built by ``prepare_insect.export_classifier_crops``:
    insect_cls/{train,val,test}/{pollinator,non_pollinator}/*.jpg
Class imbalance (pollinator >> non) is handled with a WeightedRandomSampler.

CLI:
    python -m src.cv_engine.train_classifier --freeze --epochs 12
    # server (full fine-tune, bigger backbone):
    python -m src.cv_engine.train_classifier \
        --model hf-hub:timm/eva02_large_patch14_clip_336.merged2b_ft_inat21 --epochs 15
"""

import argparse
import json
import multiprocessing
from pathlib import Path

try:
    multiprocessing.set_start_method("fork", force=True)
except (RuntimeError, ValueError):
    pass  # fork unavailable (Windows) -> use the default start method

import numpy as np
import timm
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision.datasets import ImageFolder


CLS_DIR = C.INTERIM_DIR / "insect_cls"
RUNS = C.INTERIM_DIR / "cv_runs"
# iNaturalist-2021-pretrained backbone, loaded straight from the HF Hub (the
# ``hf-hub:`` prefix is required — the tag is not in timm's local registry).
DEFAULT_MODEL = "hf-hub:timm/convnext_large_mlp.laion2b_ft_augreg_inat21"


def _loaders(model, batch):
    cfg = timm.data.resolve_data_config({}, model=model)
    train_tf = timm.data.create_transform(**cfg, is_training=True)
    eval_tf = timm.data.create_transform(**cfg, is_training=False)
    ds = {s: ImageFolder(str(CLS_DIR / s), transform=(train_tf if s == "train" else eval_tf))
          for s in ("train", "val", "test")}
    # balance the training classes with a weighted sampler
    targets = np.array([y for _, y in ds["train"].samples])
    cw = 1.0 / np.bincount(targets)
    sampler = WeightedRandomSampler(
        weights=torch.as_tensor([cw[t] for t in targets], dtype=torch.double),
        num_samples=len(targets), replacement=True)
    dl = {
        "train": DataLoader(ds["train"], batch_size=batch, sampler=sampler,
                            num_workers=6, pin_memory=True),
        "val": DataLoader(ds["val"], batch_size=batch, shuffle=False, num_workers=6, pin_memory=True),
        "test": DataLoader(ds["test"], batch_size=batch, shuffle=False, num_workers=6, pin_memory=True),
    }
    return dl, ds["train"].classes


@torch.no_grad()
def evaluate(model, loader, device) -> dict:
    model.eval()
    correct = total = 0
    per_cls_correct = [0, 0]; per_cls_total = [0, 0]
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        pred = model(x).argmax(1)
        correct += (pred == y).sum().item(); total += y.numel()
        for c in (0, 1):
            m = y == c
            per_cls_total[c] += m.sum().item()
            per_cls_correct[c] += (pred[m] == c).sum().item()
    acc = correct / max(total, 1)
    bal = 0.5 * sum(per_cls_correct[c] / max(per_cls_total[c], 1) for c in (0, 1))
    return {"acc": round(acc, 4), "balanced_acc": round(bal, 4),
            "pollinator_recall": round(per_cls_correct[0] / max(per_cls_total[0], 1), 4),
            "non_pollinator_recall": round(per_cls_correct[1] / max(per_cls_total[1], 1), 4)}


def train(model_name: str = DEFAULT_MODEL, name: str = "insect_classifier",
          epochs: int = 12, batch: int = 8, lr: float = 3e-4, freeze: bool = False) -> dict:
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = timm.create_model(model_name, pretrained=True, num_classes=2)
    if freeze:                                   # linear-probe: train the head only
        for p in model.parameters():
            p.requires_grad = False
        for p in model.get_classifier().parameters():
            p.requires_grad = True
    model.to(device)

    dl, classes = _loaders(model, batch)
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=lr, weight_decay=1e-4)
    scaler = torch.amp.GradScaler("cuda", enabled=device == "cuda")
    crit = nn.CrossEntropyLoss()

    out_dir = RUNS / name; out_dir.mkdir(parents=True, exist_ok=True)
    best_bal, best_path = -1.0, out_dir / "best.pt"
    for ep in range(1, epochs + 1):
        model.train()
        for x, y in dl["train"]:
            x, y = x.to(device), y.to(device)
            opt.zero_grad()
            with torch.amp.autocast("cuda", enabled=device == "cuda"):
                loss = crit(model(x), y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        val = evaluate(model, dl["val"], device)
        print(f"epoch {ep}/{epochs} val={val}", flush=True)
        if val["balanced_acc"] > best_bal:
            best_bal = val["balanced_acc"]
            torch.save({"model": model_name, "classes": classes,
                        "state_dict": model.state_dict()}, best_path)

    ckpt = torch.load(best_path, map_location=device)
    model.load_state_dict(ckpt["state_dict"])
    test = evaluate(model, dl["test"], device)
    result = {"model": model_name, "classes": classes, "freeze": freeze,
              "best_val_balanced_acc": best_bal, "test": test, "weights": str(best_path)}
    json.dump(result, open(out_dir / "classifier_result.json", "w"), indent=2)
    return result


def main() -> None:
    ap = argparse.ArgumentParser(description=__doc__)
    ap.add_argument("--model", default=DEFAULT_MODEL)
    ap.add_argument("--name", default="insect_classifier")
    ap.add_argument("--epochs", type=int, default=12)
    ap.add_argument("--batch", type=int, default=8)
    ap.add_argument("--lr", type=float, default=3e-4)
    ap.add_argument("--freeze", action="store_true", help="linear-probe (head only)")
    args = ap.parse_args()
    print(json.dumps(train(args.model, args.name, args.epochs, args.batch, args.lr, args.freeze), indent=2))


if __name__ == "__main__":
    main()
'''
exec(compile(_src, "src/cv_engine/train_classifier.py", "exec"), train_classifier.__dict__)
_s.modules["train_classifier"] = train_classifier
globals()["train_classifier"] = train_classifier

#### `src/cv_engine/visit_counter.py` (inlined)

In [ ]:
# ===== src/cv_engine/visit_counter.py  (inlined module — edit here) =====
import types as _t, sys as _s
visit_counter = _t.ModuleType("visit_counter")
visit_counter.C = C
_src = r'''
"""Flower-visit counting on video (two-stage insect recognition).

Pipeline:
  1. Detect flowers on **every frame** and keep stable IDs across frames via IoU
     association (handles moving camera/flowers); each flower's box is dilated
     into an approach ROI (``flower_1, flower_2, ...``).
  2. Detect and track insects across frames with a single-class YOLO26 detector
     + BoT-SORT (one track ID per insect).
  3. Classify each tracked insect crop with the iNaturalist-pretrained classifier
     as ``pollinator`` or ``non_pollinator`` (majority vote over a track's life).
  4. Count a visit whenever a tracked insect enters a flower ROI (enter-transition,
     debounced so one dwell = one visit and tracker flicker does not double-count).

Outputs:
  * ``<video>_visits.csv``  -> flower_id, total, pollinator, non_pollinator
  * annotated ``<video>_annotated.mp4`` (flowers + tracks + live counts) if --save-video

CLI:
    python -m src.cv_engine.visit_counter --video data/raw/Test_Video/clip.mp4 \
        --flower-weights .../flower/best.pt --insect-weights .../insect/best.pt \
        --classifier-weights .../insect_classifier/best.pt --save-video
"""

import argparse
import csv
from collections import Counter, defaultdict
from pathlib import Path

import cv2
import numpy as np



def _center(b):
    return (b[0] + b[2]) / 2, (b[1] + b[3]) / 2


def _in(box, pt):
    return box[0] <= pt[0] <= box[2] and box[1] <= pt[1] <= box[3]


class Classifier:
    """iNaturalist-pretrained pollinator/non_pollinator classifier (lazy torch)."""

    def __init__(self, weights: str):
        import timm
        import torch
        self.torch = torch
        ckpt = torch.load(weights, map_location="cpu", weights_only=True)
        self.classes = ckpt["classes"]
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.model = timm.create_model(ckpt["model"], pretrained=False, num_classes=len(self.classes))
        self.model.load_state_dict(ckpt["state_dict"])
        self.model.eval().to(self.device)
        cfg = timm.data.resolve_data_config({}, model=self.model)
        self.tf = timm.data.create_transform(**cfg, is_training=False)

    def predict(self, crop_bgr) -> str:
        from PIL import Image
        if crop_bgr.size == 0:
            return self.classes[0]
        img = Image.fromarray(cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB))
        x = self.tf(img).unsqueeze(0).to(self.device)
        with self.torch.no_grad():
            idx = int(self.model(x).argmax(1).item())
        return self.classes[idx]


def _iou(a, b):
    ix1, iy1 = max(a[0], b[0]), max(a[1], b[1])
    ix2, iy2 = min(a[2], b[2]), min(a[3], b[3])
    iw, ih = max(0, ix2 - ix1), max(0, iy2 - iy1)
    inter = iw * ih
    ua = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / ua if ua > 0 else 0.0


def _detect_flowers_raw(model, frame, conf, dilate=0.15):
    res = model.predict(frame, conf=conf, verbose=False)[0]
    H, W = frame.shape[:2]
    out = []
    for b in res.boxes.xyxy.cpu().numpy():
        x1, y1, x2, y2 = b
        dw, dh = (x2 - x1) * dilate, (y2 - y1) * dilate
        out.append((max(0, x1 - dw), max(0, y1 - dh), min(W, x2 + dw), min(H, y2 + dh)))
    return out


class FlowerTracker:
    """Per-frame flower detection with stable IDs across frames (IoU association).

    Handles dynamic video (moving camera/flowers): each frame the flowers are
    re-detected and matched to existing tracks by IoU so ``flower_1`` stays the
    same flower as it moves. A short ``max_missed`` grace keeps IDs through brief
    misses. ``seen`` records every flower ID ever assigned (for the final report).
    """

    def __init__(self, model, conf, dilate=0.15, iou_thr=0.3, max_missed=30):
        self.model, self.conf, self.dilate = model, conf, dilate
        self.iou_thr, self.max_missed = iou_thr, max_missed
        self.tracks: dict[str, dict] = {}
        self.next_id = 1
        self.seen: set[str] = set()

    def update(self, frame):
        dets = _detect_flowers_raw(self.model, frame, self.conf, self.dilate)
        used = set()
        for fid in list(self.tracks):
            best, bj = self.iou_thr, -1
            for j, d in enumerate(dets):
                if j in used:
                    continue
                iou = _iou(self.tracks[fid]["box"], d)
                if iou >= best:
                    best, bj = iou, j
            if bj >= 0:
                self.tracks[fid] = {"box": dets[bj], "missed": 0}
                used.add(bj)
            else:
                self.tracks[fid]["missed"] += 1
                if self.tracks[fid]["missed"] > self.max_missed:
                    del self.tracks[fid]
        for j, d in enumerate(dets):
            if j not in used:
                fid = f"flower_{self.next_id}"; self.next_id += 1
                self.tracks[fid] = {"box": d, "missed": 0}
                self.seen.add(fid)
        return self.current()

    def current(self):
        """Active flower boxes without re-detecting (reused between detect frames)."""
        return [(fid, t["box"]) for fid, t in self.tracks.items() if t["missed"] == 0]


def _load_types():
    """Map species class_id (str) -> coarse insect type from the taxonomy manifest."""
    import csv as _csv
    bee = set(_C.BEE_FAMILIES)
    out = {}
    try:
        for r in _csv.DictReader(open(_C.MANIFEST_DIR / "split_manifest.csv")):
            o, f = r["order"], r["family"]
            if f in bee: t = "bee"
            elif o == "Lepidoptera": t = "butterfly"
            elif f == "Formicidae": t = "ant"
            elif o == "Coleoptera": t = "beetle"
            elif o == "Diptera": t = "fly"
            elif o == "Hymenoptera": t = "wasp"
            elif o == "Hemiptera": t = "bug"
            else: t = (o or "insect").lower()
            out[r["class_id"]] = t
    except FileNotFoundError:
        pass
    return out


def count_visits(video, flower_weights, insect_weights, classifier_weights,
                 out_dir: Path, conf=0.1, debounce=20, save_video=False,
                 flower_interval=5, vid_stride=2) -> dict:
    from ultralytics import YOLO
    out_dir.mkdir(parents=True, exist_ok=True)
    flower_model, insect_model = YOLO(flower_weights), YOLO(insect_weights)
    classifier = Classifier(classifier_weights) if classifier_weights else None

    types = _load_types() if classifier is not None else {}
    visits = defaultdict(Counter)
    track_votes: dict[int, Counter] = defaultdict(Counter)   # track_id -> class votes
    last_flower: dict[int, str | None] = {}
    last_visit_frame: dict[tuple[int, str], int] = {}
    ftracker = FlowerTracker(flower_model, conf)
    writer = None

    stream = insect_model.track(source=video, stream=True, tracker="botsort.yaml",
                                persist=True, conf=conf, verbose=False, vid_stride=vid_stride)
    for fi, res in enumerate(stream):
        frame = res.orig_img
        # re-detect flowers every `flower_interval` frames (they move slowly);
        # reuse the tracked boxes in between -> dynamic but ~interval× faster.
        flowers = ftracker.update(frame) if fi % flower_interval == 0 else ftracker.current()
        if fi == 0 and save_video:
            h, w = frame.shape[:2]
            writer = cv2.VideoWriter(str(out_dir / (Path(video).stem + "_annotated.mp4")),
                                     cv2.VideoWriter_fourcc(*"mp4v"), 25, (w, h))
        boxes = res.boxes
        labels: dict[int, str] = {}
        if boxes is not None and boxes.id is not None:
            ids = boxes.id.int().cpu().tolist()
            xyxy = boxes.xyxy.cpu().numpy()
            for tid, b in zip(ids, xyxy):
                if classifier is not None:                    # majority vote per track
                    x1, y1, x2, y2 = map(int, b)
                    sp = classifier.predict(frame[y1:y2, x1:x2])
                    track_votes[tid][types.get(sp, sp)] += 1   # species id -> type
                    cls = track_votes[tid].most_common(1)[0][0]
                else:
                    cls = "insect"
                labels[tid] = cls
                cur = next((fid for fid, fb in flowers if _in(fb, _center(b))), None)
                if cur is not None and last_flower.get(tid) != cur:
                    if fi - last_visit_frame.get((tid, cur), -10**9) > debounce:
                        visits[cur]["total"] += 1
                        visits[cur][cls] += 1
                        last_visit_frame[(tid, cur)] = fi
                last_flower[tid] = cur
        if writer is not None:
            writer.write(_annotate(frame, flowers, res, labels, visits))
    if writer is not None:
        writer.release()

    for fid in ftracker.seen:                     # include flowers seen with 0 visits
        visits[fid]
    ctypes = sorted({t for v in visits.values() for t in v if t != "total"})
    rows = [{"flower_id": k, "total": v["total"], **{t: v.get(t, 0) for t in ctypes}}
            for k, v in sorted(visits.items())]
    csv_path = out_dir / (Path(video).stem + "_visits.csv")
    with open(csv_path, "w", newline="") as fh:
        w = csv.DictWriter(fh, fieldnames=["flower_id", "total"] + ctypes)
        w.writeheader(); w.writerows(rows)
    return {"video": Path(video).name, "flowers": len(flowers),
            "visits": {r["flower_id"]: r["total"] for r in rows}, "csv": str(csv_path)}


def _annotate(frame, flowers, res, labels, visits):
    for fid, (x1, y1, x2, y2) in flowers:
        cv2.rectangle(frame, (int(x1), int(y1)), (int(x2), int(y2)), (0, 200, 0), 2)
        cv2.putText(frame, f"{fid}:{visits[fid]['total']}", (int(x1), int(y1) - 6),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 200, 0), 2)
    if res.boxes is not None and res.boxes.id is not None:
        for tid, b in zip(res.boxes.id.int().cpu().tolist(), res.boxes.xyxy.cpu().numpy()):
            x1, y1, x2, y2 = map(int, b)
            cls = labels.get(tid, "?")
            col = (255, 120, 0) if cls == "pollinator" else (0, 0, 255)
            cv2.rectangle(frame, (x1, y1), (x2, y2), col, 2)
            cv2.putText(frame, f"{cls}#{tid}", (x1, y1 - 6),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, col, 1)
    return frame


def main() -> None:
    ap = argparse.ArgumentParser(description=__doc__)
    ap.add_argument("--video", required=True)
    ap.add_argument("--flower-weights", required=True)
    ap.add_argument("--insect-weights", required=True)
    ap.add_argument("--classifier-weights", default="")
    ap.add_argument("--out", default=str(C.INTERIM_DIR / "cv_runs" / "visits"))
    ap.add_argument("--conf", type=float, default=0.1)
    ap.add_argument("--debounce", type=int, default=20)
    ap.add_argument("--flower-interval", type=int, default=5,
                    help="re-detect flowers every N frames (dynamic video)")
    ap.add_argument("--vid-stride", type=int, default=2,
                    help="process every Nth frame (lower effective fps, stabler tracks)")
    ap.add_argument("--save-video", action="store_true")
    args = ap.parse_args()
    import json
    print(json.dumps(count_visits(args.video, args.flower_weights, args.insect_weights,
                                  args.classifier_weights, Path(args.out), args.conf,
                                  args.debounce, args.save_video, args.flower_interval,
                                  args.vid_stride), indent=2))


if __name__ == "__main__":
    main()
'''
exec(compile(_src, "src/cv_engine/visit_counter.py", "exec"), visit_counter.__dict__)
_s.modules["visit_counter"] = visit_counter
globals()["visit_counter"] = visit_counter

## 1 · Flower detector
Build the single-class flower detection set (GrabCut auto-boxes) and fine-tune YOLO26.

In [ ]:
flower_w = RUNS / "flower_yolo26n" / "weights" / "best.pt"
if not exists(flower_w):
    if not exists(C.INTERIM_DIR / "flower_det" / "data.yaml"):
        print(prepare_flower.run(workers=os.cpu_count())["total"], "flower images prepared")
    cvtrain.train(str(C.INTERIM_DIR / "flower_det" / "data.yaml"), "flower_yolo26n", epochs=60)
else:
    print("flower detector already trained — skipping")
print("flower weights:", flower_w)

## 2 · Insect detector + pollinator classifier
Build the insect datasets (iNat auto-boxed + BEE.v8i real boxes), train the single-class detector, then the iNaturalist-pretrained classifier.

In [ ]:
insect_w = RUNS / "insect1cls_yolo26n" / "weights" / "best.pt"
clf_w = RUNS / "insect_classifier" / "best.pt"

if not exists(C.INTERIM_DIR / "insect_det1"):
    prepare_insect.run(); prepare_insect.export_single_class(); prepare_insect.export_classifier_crops()
    print("insect datasets built")

if not exists(insect_w):
    cvtrain.train(str(C.INTERIM_DIR / "insect_det1" / "data.yaml"), "insect1cls_yolo26n",
                  epochs=60, scale=0.9)
else:
    print("insect detector already trained — skipping")

if not exists(clf_w):
    train_classifier.train(name="insect_classifier", epochs=8, batch=8, freeze=True)
else:
    print("classifier already trained — skipping")
print("insect weights:", insect_w, "| classifier:", clf_w)

## 3 · Metrics

In [ ]:
def best_map(csv_path):
    df = pd.read_csv(csv_path); return round(df["metrics/mAP50(B)"].max(), 4)
metrics = {
    "flower_detector_mAP50": best_map(RUNS / "flower_yolo26n" / "results.csv"),
    "insect_detector_mAP50": best_map(RUNS / "insect1cls_yolo26n" / "results.csv"),
    "classifier": json.load(open(RUNS / "insect_classifier" / "classifier_result.json"))["test"],
}
print(json.dumps(metrics, indent=2))

## 4 · Tracking + flower-visit counting
Run every test video through detect → track → classify → count. Flowers are detected per-frame with stable IDs; an insect entering a flower ROI is one visit.

In [ ]:
out_dir = RUNS / "visits"
videos = sorted((C.TEST_VIDEO_DIR).glob("*.mp4"))
results = []
for v in videos:
    r = visit_counter.count_visits(str(v), str(flower_w), str(insect_w), str(clf_w),
                                   out_dir, save_video=True, flower_interval=5)
    results.append(r); print(v.name, "->", r["flowers"], "flowers,", sum(r["visits"].values()), "visits")

### Visit tallies

In [ ]:
frames = []
for csvf in sorted(out_dir.glob("*_visits.csv")):
    d = pd.read_csv(csvf); d.insert(0, "video", csvf.stem.replace("_visits", ""))
    frames.append(d)
visits_df = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
visits_df[visits_df["total"] > 0] if not visits_df.empty else visits_df

### Sample annotated frame

In [ ]:
import cv2, glob
vids = sorted(glob.glob(str(out_dir / "*_annotated.mp4")), key=os.path.getsize)
if vids:
    cap = cv2.VideoCapture(vids[-1]); cap.set(cv2.CAP_PROP_POS_FRAMES, int(cap.get(cv2.CAP_PROP_FRAME_COUNT) * 0.5))
    ok, fr = cap.read(); cap.release()
    if ok:
        plt.figure(figsize=(11, 6)); plt.imshow(cv2.cvtColor(fr, cv2.COLOR_BGR2RGB)); plt.axis("off")
        plt.title("flowers (green) + insect tracks (bee=orange / non=red)"); plt.show()

## 5 · Summary
All trained weights are under `data/interim/cv_runs/*/weights/`; annotated videos + visit CSVs under `data/interim/cv_runs/visits/`.

In [ ]:
print(json.dumps({"metrics": metrics,
                  "videos_processed": len(results),
                  "outputs": str(out_dir)}, indent=2))